# Email Header Phishing Detector

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import re
import pandas as pd
import email
from email import policy
import ipaddress
import requests

sample_headers = [
    """Received: from suspicious.domain.com (suspicious.domain.com. [203.0.113.5])
        by mail.example.com with ESMTPS id abc123
        for <victim@example.com>;
        Tue, 25 May 2025 10:23:45 -0700 (PDT)
    From: "PayPal Security" <service@paypal-fraud-check.com>
    Reply-To: fraud-check@fakepaypal.com
    Subject: Urgent: Account Verification Needed
    Return-Path: <fraud-check@fakepaypal.com>
    Received-SPF: fail (google.com: domain of fakepaypal.com does not designate 203.0.113.5 as permitted sender)
    Authentication-Results: mail.example.com; dkim=fail; spf=fail; dmarc=fail""",

    """Received: from mail.google.com (mail.google.com. [209.85.220.41])
        by mx.example.com with ESMTPS id def456
        for <user@example.com>;
        Mon, 24 May 2025 14:18:32 -0700 (PDT)
    From: "Google Alerts" <alerts-noreply@google.com>
    Subject: Your Daily Alert
    Return-Path: <alerts-noreply@google.com>
    Received-SPF: pass (google.com: domain of google.com designates 209.85.220.41 as permitted sender)
    Authentication-Results: mx.example.com; dkim=pass; spf=pass; dmarc=pass"""
]

def extract_field(header, field):
    pattern = re.compile(f"{field}:(.*)", re.IGNORECASE)
    match = pattern.search(header)
    return match.group(1).strip() if match else ""

def extract_received_ip(header):
    matches = re.findall(r"\[([\d\.]+)\]", header)
    return matches[0] if matches else None

def geolocate_ip(ip):
    try:
        r = requests.get(f"https://ipapi.co/{ip}/country_name/", timeout=3)
        return r.text.strip()
    except:
        return "Unknown"

def analyze_header(header):
    score = 0
    from_field = extract_field(header, "From")
    reply_to = extract_field(header, "Reply-To")
    return_path = extract_field(header, "Return-Path")
    received_ip = extract_received_ip(header)
    spf_result = re.search(r"Received-SPF:\s*(\w+)", header, re.IGNORECASE)
    dkim_result = re.search(r"dkim=(\w+)", header, re.IGNORECASE)
    dmarc_result = re.search(r"dmarc=(\w+)", header, re.IGNORECASE)

    if reply_to and reply_to not in from_field:
        score += 1
    if return_path and return_path not in from_field:
        score += 1
    if spf_result and spf_result.group(1).lower() != "pass":
        score += 1
    if dkim_result and dkim_result.group(1).lower() != "pass":
        score += 1
    if dmarc_result and dmarc_result.group(1).lower() != "pass":
        score += 1
    location = geolocate_ip(received_ip) if received_ip else "Unknown"
    if location not in ["United States", "Unknown"]:
        score += 1

    return {
        "From": from_field,
        "Reply-To": reply_to,
        "Return-Path": return_path,
        "Received IP": received_ip,
        "Geo": location,
        "SPF": spf_result.group(1) if spf_result else "None",
        "DKIM": dkim_result.group(1) if dkim_result else "None",
        "DMARC": dmarc_result.group(1) if dmarc_result else "None",
        "Phishing Score": score,
        "Verdict": "Phishing" if score >= 3 else "Suspicious" if score == 2 else "Likely Safe"
    }

results = [analyze_header(h) for h in sample_headers]
df = pd.DataFrame(results)
df
